# Estrazione URL di progetto dai JSON

Legge i JSON prodotti da GROBID, estrae gli URL dalle footnotes e usa
Ollama (llama3.1:8b su GPU T4) per identificare l'URL del sito del progetto.

Per ogni URL trovato viene effettuato un fetch HTTP per estrarne il contenuto testuale,
che viene poi passato al modello per una valutazione più accurata.

**Prima di iniziare:** assicurati di avere il runtime GPU attivo:
Runtime > Cambia tipo di runtime > T4 GPU

## Cella 1 — Installa Ollama e avvia il server

In [ ]:
%%capture
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama tqdm requests beautifulsoup4 lxml

In [ ]:
import subprocess, time, requests

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

try:
    requests.get('http://localhost:11434', timeout=5)
    print('OK: Ollama server attivo')
except Exception:
    print('ERRORE: Ollama non risponde, riesegui questa cella')

## Cella 2 — Scarica il modello

In [ ]:
# Su T4 (15GB VRAM) llama3.1:8b gira comodamente
# Download circa 4.7GB, circa 2 minuti
!ollama pull llama3.1:8b
print('OK: Modello pronto')

## Cella 3 — Carica i JSON

In [ ]:
from google.colab import files
import json

print('Carica i file article_*.json (puoi selezionarne piu di uno)...')
uploaded = files.upload()

articles = []
for filename, content in uploaded.items():
    try:
        article = json.loads(content.decode('utf-8'))
        article['_filename'] = filename
        articles.append(article)
    except json.JSONDecodeError:
        print(f'SKIP: {filename} non e un JSON valido')

articles.sort(key=lambda a: a['_filename'])

print(f'Caricati {len(articles)} articoli:')
for a in articles:
    fn_count = len(a.get('footnotes') or [])
    print(f"  {a['_filename']} | {a.get('title', '(no title)')[:60]} | {fn_count} footnotes")

## Cella 4 — Funzioni di fetch del contenuto degli URL

Per ogni URL viene fatto un fetch HTTP e il testo della pagina viene estratto
e troncato a ~1500 caratteri per non sovraccaricare il context del modello.

In [ ]:
import requests as req
from bs4 import BeautifulSoup

FETCH_TIMEOUT = 10        # secondi per singola richiesta HTTP
MAX_CONTENT_CHARS = 1500  # caratteri estratti per pagina, passati al modello

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; ResearchBot/1.0)'
}

def fetch_url_content(url: str) -> dict:
    """
    Fetcha una URL e restituisce un dict con:
      - status: 'ok' | 'error' | 'timeout' | 'skip'
      - http_code: int o None
      - content: testo estratto (troncato) o messaggio di errore
    """
    # Salta URL non-web (DOI resolver, GitHub raw, PDF diretti, ecc.)
    skip_patterns = [
        'doi.org', 'raw.githubusercontent.com',
        '.pdf', '.zip', '.xml', '.json'
    ]
    if any(p in url.lower() for p in skip_patterns):
        return {'status': 'skip', 'http_code': None, 'content': '[skipped: non-HTML resource]'}

    try:
        response = req.get(url, timeout=FETCH_TIMEOUT, headers=HEADERS,
                           allow_redirects=True)
        http_code = response.status_code

        if http_code != 200:
            return {'status': 'error', 'http_code': http_code,
                    'content': f'[HTTP {http_code}]'}

        content_type = response.headers.get('Content-Type', '')
        if 'html' not in content_type:
            return {'status': 'skip', 'http_code': http_code,
                    'content': f'[non-HTML content-type: {content_type}]'}

        soup = BeautifulSoup(response.text, 'lxml')

        # Rimuovi script e stili
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()

        # Estrai testo pulito
        text = soup.get_text(separator=' ', strip=True)
        text = ' '.join(text.split())  # normalizza spazi
        text = text[:MAX_CONTENT_CHARS]

        return {'status': 'ok', 'http_code': 200, 'content': text}

    except req.exceptions.Timeout:
        return {'status': 'timeout', 'http_code': None, 'content': '[timeout]'}
    except Exception as e:
        return {'status': 'error', 'http_code': None, 'content': f'[error: {str(e)[:80]}]'}


def fetch_all_urls(urls: list, verbose: bool = True) -> dict:
    """
    Fetcha una lista di URL e restituisce un dict {url: fetch_result}.
    Stampa un breve log se verbose=True.
    """
    results = {}
    for url in urls:
        result = fetch_url_content(url)
        results[url] = result
        if verbose:
            icon = {'ok': '✓', 'error': '✗', 'timeout': '⏱', 'skip': '—'}.get(result['status'], '?')
            code = f" [{result['http_code']}]" if result['http_code'] else ''
            print(f"    {icon}{code} {url[:80]}")
    return results

print('OK: Funzioni di fetch pronte')

## Cella 5 — Estrazione URL + fetch contenuto + chiamata Ollama

In [ ]:
import re
import json
import ollama
from tqdm.notebook import tqdm

OLLAMA_MODEL = 'llama3.1:8b'

URL_PATTERN = re.compile(r'https?://[^\s\)\]\},"\'>]+', re.IGNORECASE)

SYSTEM_PROMPT = (
    "You are an expert analyst of academic papers in digital humanities.\n"
    "Your task: given a paper's title, abstract, and a list of URLs found in its footnotes "
    "(each with a snippet of their actual web content), "
    "identify which URL is the official website of the research PROJECT described in the paper.\n\n"
    "Rules:\n"
    "- Return ONLY valid JSON, no markdown, no explanation.\n"
    "- A project website is a page CREATED AND MAINTAINED BY THE PROJECT TEAM to present the project itself — "
    "not a page that merely mentions or references the project.\n"
    "- The URL must point to the project's own dedicated online presence: its homepage, portal, or interface.\n"
    "- CRITICALLY, the URL must provide access to actual project OUTPUT or DATA, such as:\n"
    "  * Digital editions or critical editions of texts\n"
    "  * Digital libraries or archives with browsable/searchable content\n"
    "  * Datasets or corpora produced by the project\n"
    "  * Annotated collections, databases, or catalogues\n"
    "  * Interactive portals giving access to the project's resources\n"
    "- USE THE PAGE CONTENT SNIPPETS to verify whether a URL actually contains project data/outputs "
    "rather than just describing them.\n"
    "- EXCLUDE the following types of URLs:\n"
    "  * Pages that only DESCRIBE what the project has done, without providing access to the actual outputs\n"
    "  * Research center or university institutional pages\n"
    "  * Tools, libraries, or software frameworks used by the project (e.g. EVT, Transkribus)\n"
    "  * Standards or specification pages (e.g. IIIF, XML TEI, Dublin Core)\n"
    "  * 'Showcase' or aggregator pages (e.g. project directories, conference listings, blog posts about the project)\n"
    "  * DOI links, Wikipedia pages, or generic academic repositories\n"
    "  * GitHub repos (acceptable ONLY if no dedicated website exists AND the repo contains the actual project data/edition)\n"
    "- If multiple URLs seem plausible, prefer the one that gives direct access to the project's data or digital edition "
    "over one that merely describes the project.\n"
    "- If no URL fits, return null.\n\n"
    'Response format (strictly):\n'
    '{"project_url": "https://example.com"}\n'
    '{"project_url": null}'
)


def extract_urls(footnotes):
    seen, urls = set(), []
    for fn in (footnotes or []):
        for url in URL_PATTERN.findall(fn):
            url = url.rstrip('.,;:)')
            if url not in seen:
                seen.add(url)
                urls.append(url)
    return urls


def ask_ollama(title, abstract, urls, fetched_contents):
    """Chiama Ollama passando titolo, abstract e URL con i loro contenuti fetchati."""
    if not urls:
        return None

    url_blocks = []
    for url in urls:
        info = fetched_contents.get(url, {})
        status = info.get('status', 'unknown')
        content = info.get('content', '')
        if status == 'ok':
            snippet = content[:600]  # ulteriore troncamento nel prompt
            url_blocks.append(f"  URL: {url}\n  Page content snippet: {snippet}")
        else:
            url_blocks.append(f"  URL: {url}\n  Page content snippet: {content}")

    url_section = '\n\n'.join(url_blocks)

    prompt = (
        f"Title: {title or '(no title)'}\n\n"
        f"Abstract: {abstract or '(no abstract)'}\n\n"
        f"URLs from footnotes (with page content snippets):\n\n{url_section}\n\n"
        "Which URL is the official project website?"
    )

    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt},
            ],
            options={'temperature': 0},
        )
    except Exception as e:
        print(f'  ERRORE Ollama: {e}')
        return None

    raw = response['message']['content'].strip()
    match = re.search(r'\{[^{}]*"project_url"[^{}]*\}', raw, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group()).get('project_url')
    except json.JSONDecodeError:
        return None


# --- Esegui ---

results = []
stats = {'trovati': 0, 'non_trovati': 0, 'saltati': 0}

for article in tqdm(articles, desc='Analisi articoli'):
    title     = article.get('title') or ''
    abstract  = article.get('abstract') or ''
    footnotes = article.get('footnotes') or []
    filename  = article.get('_filename', '')

    urls = extract_urls(footnotes)

    if not urls:
        stats['saltati'] += 1
        continue

    print(f'\n{filename}')
    print(f'  Titolo: {title[:70]}')
    print(f'  URL trovati: {len(urls)}')

    # Fetch del contenuto di ogni URL
    print(f'  Fetch contenuti:')
    fetched = fetch_all_urls(urls, verbose=True)
    ok_count = sum(1 for v in fetched.values() if v['status'] == 'ok')
    print(f'  Fetch OK: {ok_count}/{len(urls)}')

    # Chiamata al modello con i contenuti
    project_url = ask_ollama(title, abstract, urls, fetched)

    if project_url:
        stats['trovati'] += 1
        print(f'  -> Progetto: {project_url}')
    else:
        stats['non_trovati'] += 1
        print(f'  -> Nessun URL di progetto identificato')

    results.append({
        'file':           filename,
        'title':          title,
        'project_url':    project_url,
        'all_urls_found': urls,
        'fetch_summary':  {url: {'status': v['status'], 'http_code': v['http_code']}
                           for url, v in fetched.items()},
    })

print(f'\nAnalizzati:   {len(results)}')
print(f'Con URL:      {stats["trovati"]}')
print(f'Senza URL:    {stats["non_trovati"]}')
print(f'Saltati:      {stats["saltati"]} (nessun URL nelle footnotes)')

## Cella 6 — Salva e scarica i risultati

In [ ]:
from google.colab import files
import json

output_path = '/content/project_urls.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'Salvato: {output_path}')
files.download(output_path)